# RetainIQ — Phase 3.1: MySQL Data Model & Star Schema

## Objective

In this notebook, I design the MySQL analytical model for RetainIQ. I am moving from the cleaned customer-level dataset into a relational analytical structure that supports SQL analysis, Power BI, customer segmentation, revenue-at-risk analysis, and later AI/RAG workflows.

My design preserves the **one-row-per-customer grain** established in Phase 1 and Phase 2.

## 1. Load the Cleaned Dataset and Prepare MySQL Access

In [2]:
from pathlib import Path
from getpass import getpass
import pandas as pd
import mysql.connector
from mysql.connector import Error

pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_clean.csv")
if not DATA_PATH.exists(): DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_clean.csv")
if not DATA_PATH.exists(): DATA_PATH = Path("C:\\RetainIQ — AI-Powered Telecom Customer Retention Intelligence Platform\\RetainIQ_phase_02_data_cleaning_full\\data\\telco_clean.csv")
df = pd.read_csv(DATA_PATH)
print(f"Loaded cleaned dataset: {df.shape[0]:,} rows × {df.shape[1]:,} columns")

MYSQL_CONFIG = {"host":"localhost","port":3306,"user":"retainiq_user","password":getpass("Enter MySQL password for retainiq_user: "),"database":"retainiq"}

def get_connection(): return mysql.connector.connect(**MYSQL_CONFIG)

def run_query(query, params=None):
    connection=cursor=None
    try:
        connection=get_connection(); cursor=connection.cursor(dictionary=True); cursor.execute(query, params or ()); return pd.DataFrame(cursor.fetchall())
    except Error as exc:
        print(f"MySQL error: {exc}"); return None
    finally:
        if cursor: cursor.close()
        if connection and connection.is_connected(): connection.close()

print("MySQL helper functions are ready.")

Loaded cleaned dataset: 7,043 rows × 51 columns
MySQL helper functions are ready.


## 2. Validate the Analytical Grain

In [3]:
grain_audit = pd.Series({"Rows":len(df),"Columns":len(df.columns),"Unique Customer IDs":df["Customer ID"].nunique(),"Duplicate Customer IDs":int(df["Customer ID"].duplicated().sum()),"Missing Customer IDs":int(df["Customer ID"].isna().sum())})
grain_audit

Rows                      7043
Columns                     51
Unique Customer IDs       7043
Duplicate Customer IDs       0
Missing Customer IDs         0
dtype: int64

In [4]:
assert df["Customer ID"].notna().all()
assert df["Customer ID"].is_unique
assert len(df)==7043
assert len(df.columns)==51
print("PASS — I confirmed the Phase 2 dataset and one-row-per-customer grain.")

PASS — I confirmed the Phase 2 dataset and one-row-per-customer grain.


## 3. Fact Table Design

I will use `fact_customer_status` for customer-level measures and core outcomes.

### Grain
**One row = one customer**

### Fields
`customer_id`, `tenure_months`, `monthly_charge`, `total_charges`, `total_refunds`, `total_revenue`, `satisfaction_score`, `churn_score`, `cltv`, `churn_label`, `customer_status`

In [5]:
fact_mapping=pd.DataFrame({"Source Column":["Customer ID","Tenure in Months","Monthly Charge","Total Charges","Total Refunds","Total Revenue","Satisfaction Score","Churn Score","CLTV","Churn Label","Customer Status"],"MySQL Column":["customer_id","tenure_months","monthly_charge","total_charges","total_refunds","total_revenue","satisfaction_score","churn_score","cltv","churn_label","customer_status"]})
fact_mapping

,Source Column,MySQL Column
0,Customer ID,customer_id
1,Tenure in Months,tenure_months
2,Monthly Charge,monthly_charge
3,Total Charges,total_charges
4,Total Refunds,total_refunds
5,Total Revenue,total_revenue
6,Satisfaction Score,satisfaction_score
7,Churn Score,churn_score
8,CLTV,cltv
9,Churn Label,churn_label


## 4. Dimension Design

I separate descriptive fields by business role.

| Dimension | Purpose |
|---|---|
| `dim_demographics` | Customer and household characteristics |
| `dim_location` | Geographic attributes |
| `dim_services` | Telecom service adoption and usage |
| `dim_account` | Contract, offer, billing, and payment attributes |
| `dim_churn_detail` | Churn category and churn reason |

## 5. Star Schema

```text
                         dim_demographics
                               │
                               │ customer_id
                               ▼
dim_location ───────► fact_customer_status ◄────── dim_services
                               ▲
                               │
                    ┌──────────┴──────────┐
                    │                     │
              dim_account          dim_churn_detail
```

I use `customer_id` as the shared natural key because it was validated as unique.

## 6. MySQL Data-Type Strategy

I use `VARCHAR` for categories/identifiers, `SMALLINT`/`TINYINT` for compact numerics, `INT` for counts/CLTV, `DECIMAL` for money, `BOOLEAN` for `is_new_customer`, and `DECIMAL(9,6)` for coordinates.

## 7. Staging Architecture

```text
telco_clean.csv
      │
      ▼
stg_telco_clean
      │
      ├───────────────┐
      ▼               ▼
fact_customer_status   dimensions
```

The staging layer gives me a controlled landing area before I distribute records into analytical tables.

## 8. Keys and Constraints

I will enforce a primary key on `customer_id` in the fact and dimensions, foreign keys back to the fact, `NOT NULL` on core fact measures, and InnoDB for relational integrity.

# Phase 3.1 Conclusion

I have completed the conceptual and logical design of the MySQL analytical model.

**Next:** `02_mysql_database_build_and_load.ipynb`